# 02 — Semantic Matching with Sentence Embeddings

This notebook demonstrates how the **Resume Analyzer** uses dense vector embeddings
and cosine similarity to match candidates to job descriptions.

### What you will learn:
1. **Embedding representation**: How text converts into 384-dimensional dense vectors using `all-MiniLM-L6-v2`.
2. **Phrase-pair similarity**: Comparing synonymous, related, and completely unrelated phrases.
3. **Sentence-to-sentence similarity matrix**: Visualizing cross-sentence similarity as an annotated heatmap.
4. **Chunk-level best-match algorithm**: Why matching individual JD requirements against resume sentences outperforms naive whole-document pooling.

In [ ]:
# ── Cell 1: Setup & Imports ──────────────────────────────────────────────────
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is in sys.path
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.matcher import (
    get_model,
    split_sentences,
    compute_similarity_matrix,
    semantic_similarity,
)

print('✅ Environment ready. Project root:', PROJECT_ROOT)

---
## 1. Inspecting the Embedding Model

We use `all-MiniLM-L6-v2` from HuggingFace Sentence Transformers.
- **Embedding dimension**: 384 numbers per sentence
- **Normalized vectors**: When embeddings are L2-normalized ($||v|| = 1$), the cosine similarity between two vectors $u$ and $v$ is simply their dot product: $\cos(u, v) = u \cdot v$.

In [ ]:
# ── Cell 2: Load Model & Encode Sample ──────────────────────────────────────
model = get_model()
sample_phrase = "Backend engineer building scalable microservices with FastAPI and PostgreSQL."
vector = model.encode(sample_phrase, normalize_embeddings=True)

print(f"Phrase: '{sample_phrase}'")
print(f"Vector shape: {vector.shape}")
print(f"L2 Norm: {np.linalg.norm(vector):.4f} (normalized)")
print(f"First 10 values:", np.round(vector[:10], 4))

---
## 2. Phrase-Pair Similarity Experiments

Let's test semantic similarity across 3 distinct tiers:
- **Tier 1 (Synonyms / Paraphrases)**: High similarity (~0.75 - 0.95)
- **Tier 2 (Related domains)**: Moderate similarity (~0.35 - 0.65)
- **Tier 3 (Unrelated domains)**: Low similarity (< 0.25)

In [ ]:
# ── Cell 3: Phrase Pair Tests ────────────────────────────────────────────────
pairs = [
    # High similarity (same meaning, different words)
    ("Python backend engineer", "Python web developer", "Synonym / Alternate Title"),
    ("Proficient in Kubernetes and Docker", "Experience containerizing applications with Docker & K8s", "Containerization Paraphrase"),
    ("Designed RESTful APIs using FastAPI", "Built REST endpoints with FastAPI framework", "API Development"),
    
    # Moderate similarity (related technical domains)
    ("PostgreSQL database query optimization", "Backend REST API performance tuning", "Related Backend Performance"),
    ("Machine learning engineer with PyTorch", "Data analyst proficient in Excel and SQL", "Related Tech Fields"),
    ("DevOps and CI/CD pipelines", "Backend system architecture", "Related Engineering"),
    
    # Low similarity (unrelated or different industries)
    ("Senior Python software engineer", "Certified corporate tax accountant", "Unrelated Industry"),
    ("Machine learning engineer with PyTorch", "Creative director and graphic designer", "Unrelated Role"),
    ("Kubernetes cluster administration", "Customer service representative", "Completely Unrelated"),
]

results = []
for p1, p2, category in pairs:
    emb1 = model.encode(p1, normalize_embeddings=True)
    emb2 = model.encode(p2, normalize_embeddings=True)
    cos_sim = float(np.dot(emb1, emb2))
    results.append({
        "Phrase 1": p1,
        "Phrase 2": p2,
        "Category": category,
        "Cosine Similarity": round(cos_sim, 4),
    })

df_pairs = pd.DataFrame(results)
df_pairs

In [ ]:
# ── Cell 4: Bar Plot of Phrase Similarities ─────────────────────────────────
plt.figure(figsize=(10, 6), dpi=100)
colors = ['#2ecc71' if s >= 0.7 else '#f1c40f' if s >= 0.35 else '#e74c3c' for s in df_pairs['Cosine Similarity']]

bars = plt.barh(df_pairs['Category'], df_pairs['Cosine Similarity'], color=colors, height=0.6)
plt.xlabel('Cosine Similarity', fontsize=12, fontweight='bold')
plt.title('Semantic Similarity Across Phrase Pairs (all-MiniLM-L6-v2)', fontsize=14, fontweight='bold', pad=15)
plt.xlim(0, 1.0)
plt.grid(axis='x', linestyle='--', alpha=0.5)

for bar in bars:
    width = bar.get_width()
    plt.text(width + 0.02, bar.get_y() + bar.get_height() / 2, f'{width:.2f}',
             va='center', ha='left', fontsize=10, fontweight='bold')

plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

---
## 3. Real-World Resume vs Job Description Heatmap

When comparing a full resume to a job description:
- A resume may be 50+ lines long with contact info, education, varied experiences.
- A job description lists 4–8 specific requirements.

**Whole-document pooling** compresses everything into one vector, which causes dilute scores.
**Sentence-level matching** builds a similarity matrix and checks whether **each JD requirement** has a strong match somewhere in the resume.

In [ ]:
# ── Cell 5: Sample Job Description & Resume ──────────────────────────────────
sample_jd = """
Required Qualifications:
- 3+ years experience with Python and backend frameworks like FastAPI or Django.
- Strong background with PostgreSQL relational database design and SQL queries.
- Experience containerizing microservices using Docker and managing Kubernetes deployments.
- Hands-on practice designing automated CI/CD deployment pipelines on AWS.
- Excellent cross-functional team communication and mentoring skills.
"""

sample_resume = """
Alex Mercer - Senior Software Developer
- Over 4 years developing high-throughput web applications with Python and FastAPI.
- Architected PostgreSQL schemas and optimized indexing to reduce query latency by 45%.
- Packaged applications into Docker containers and automated deployments on Kubernetes clusters.
- Set up GitHub Actions CI/CD pipelines to build and deploy test containers to AWS EC2.
- Mentored 3 junior developers and collaborated closely with product managers on feature delivery.
- Familiar with React, TypeScript, TailwindCSS for internal admin tool development.
- Avid cyclist and landscape photography enthusiast.
"""

jd_sents = split_sentences(sample_jd)
resume_sents = split_sentences(sample_resume)

print(f"Extracted {len(jd_sents)} JD requirement sentences.")
print(f"Extracted {len(resume_sents)} Resume experience sentences.")

In [ ]:
# ── Cell 6: Cosine Similarity Matrix Heatmap ────────────────────────────────
sim_matrix = compute_similarity_matrix(jd_sents, resume_sents)

# Short labels for visualization
jd_labels = [f"JD {i+1}: " + (s[:35] + '...' if len(s) > 35 else s) for i, s in enumerate(jd_sents)]
res_labels = [f"Res {j+1}: " + (s[:30] + '...' if len(s) > 30 else s) for j, s in enumerate(resume_sents)]

plt.figure(figsize=(12, 6), dpi=100)
sns.heatmap(
    sim_matrix,
    annot=True,
    fmt=".2f",
    cmap="YlGnBu",
    xticklabels=res_labels,
    yticklabels=jd_labels,
    cbar_kws={'label': 'Cosine Similarity'},
    linewidths=0.5,
)

plt.title("Sentence-to-Sentence Similarity Matrix (JD vs Resume)", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Candidate Resume Sentences", fontsize=11, fontweight='bold')
plt.ylabel("Job Description Requirements", fontsize=11, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 7: Best-Match Alignment & Aggregate Score ──────────────────────────
details = semantic_similarity(sample_resume, sample_jd, return_details=True)

print("=" * 75)
print(f"OVERALL SEMANTIC MATCH SCORE: {details['score'] * 100:.1f}%")
print("=" * 75)
print("\nRequirement-by-Requirement Alignment:")

for item in details['best_matches']:
    sim = item['similarity']
    status = "✅ Strong Match" if sim >= 0.70 else "🟡 Moderate Match" if sim >= 0.50 else "❌ Weak Match"
    print(f"\n[Score: {sim:.2f} | {status}]")
    print(f"  JD Req : {item['jd_sentence']}")
    print(f"  Best Res: {item['best_resume_sentence']}")